# Mooncake Writer Demo

This notebook demonstrates how to capture **mooncake traces** — hash-block representations of LLM prompts that model KV-cache prefix sharing.

You will learn how to:
- Convert text into a sequence of hash block IDs
- Reconstruct text from those hash IDs
- Process batches of texts and observe shared prefix hashes
- Detect shared prefixes across separate requests
- Capture timestamped traces and write aiperf-compatible JSONL files

In [1]:
from mooncake_writer import MooncakeWriter

## 1. Initialise the Writer

`MooncakeWriter` needs a tokenizer to split text into tokens before hashing.
Pass any HuggingFace model name — the tokenizer is loaded via `aiperf`'s `Tokenizer.from_pretrained`.
The default `block_size` of **512 tokens** controls how many tokens are grouped into each hash block.

In [2]:
writer = MooncakeWriter("gpt2")
print(f"Block size: {writer.block_size} tokens")

Block size: 512 tokens


## 2. Convert Text to Hash Blocks

Given a text string, `text_to_hashes` will:
1. Tokenize the text
2. Split the token sequence into fixed-size blocks
3. Assign each block a unique integer hash ID using a rolling hash

The result is a compact list of integers that represents the original text.
Texts that share a common prefix will share the same leading hash IDs — this is how KV-cache prefix overlap (cache hits) is modelled.

In [3]:
text = "Hello, world! This is a test of the mooncake writer."
hash_ids = writer.text_to_hashes(text)

print(f"Text:      {text}")
print(f"Hash IDs:  {hash_ids}")
print(f"Blocks:    {len(hash_ids)}")

Text:      Hello, world! This is a test of the mooncake writer.
Hash IDs:  [0]
Blocks:    1


## 3. Reconstruct Text from Hashes

`hashes_to_text` converts hash IDs back into a text string.
It uses a `PromptGenerator` backed by a cached corpus (Shakespeare) so that the same hash ID always produces the same token block.
You need to provide `input_length` — the target length in tokens — so the writer knows how many tokens to generate.

In [4]:
input_length = len(writer.tokenizer.encode(text))
reconstructed = writer.hashes_to_text(hash_ids, input_length=input_length)

print(f"Original:      {text}")
print(f"Reconstructed: {reconstructed}")
print(f"Token length:  {input_length}")

Original:      Hello, world! This is a test of the mooncake writer.
Reconstructed: <|endoftext|> be rich no more So shall thou feed on death that feeds on
Token length:  14


## 4. Batch Operations and Prefix Sharing

In real workloads you often have many prompts. `texts_to_hashes` processes a list of texts at once.

Because the rolling hash is shared across the batch, **texts that begin with the same tokens will receive the same leading hash IDs**.
This directly models KV-cache prefix hits — if two requests share a prefix, the cache blocks for that prefix are reused.

In [7]:
writer.reset_hashes()

texts = [
    "The quick brown fox jumps over the lazy dog.",
    "The quick brown fox jumps over the lazy dog.",
    "A completely different sentence with no shared prefix.",
]

hash_sequences = writer.texts_to_hashes(texts)

for i, (t, h) in enumerate(zip(texts, hash_sequences)):
    print(f"Text {i+1}: {t}")
    print(f"  Hashes: {h}")
    print()

Text 1: The quick brown fox jumps over the lazy dog.
  Hashes: [0]

Text 2: The quick brown fox jumps over the lazy dog.
  Hashes: [0]

Text 3: A completely different sentence with no shared prefix.
  Hashes: [1]



## 5. Batch Reconstruction

`hashes_to_texts` reverses the batch conversion.
Each hash ID sequence is turned back into text using the same deterministic corpus mapping.

In [9]:
input_lengths = [len(writer.tokenizer.encode(t)) for t in texts]
reconstructed_texts = writer.hashes_to_texts(hash_sequences, input_lengths)

for i, (original, recon) in enumerate(zip(texts, reconstructed_texts)):
    print(f"Text {i+1}")
    print(f"  Original:      {original}")
    print(f"  Reconstructed: {recon}")
    print()

Text 1
  Original:      The quick brown fox jumps over the lazy dog.
  Reconstructed: <|endoftext|> be rich no more So shall thou feed on death that feeds on

Text 2
  Original:      The quick brown fox jumps over the lazy dog.
  Reconstructed: <|endoftext|> be rich no more So shall thou feed on death that feeds on

Text 3
  Original:      A completely different sentence with no shared prefix.
  Reconstructed: <|endoftext|> lank and lean with thy extort



## 6. Custom Block Size

You can override the block size per call to control the granularity of hashing.
Smaller blocks produce more hash IDs per text, giving finer-grained prefix sharing at the cost of more IDs to track.

In [18]:
text = "A longer piece of text to demonstrate how block size affects the number of hash blocks produced. A longer piece of text to demonstrate how block size affects the number of hash blocks produced. A longer piece of text to demonstrate how block size affects the number of hash blocks produced. A longer piece of text to demonstrate how block size affects the number of hash blocks produced. A longer piece of text to demonstrate how block size affects the number of hash blocks produced. A longer piece of text to demonstrate how block size affects the number of hash blocks produced.A longer piece of text to demonstrate how block size affects the number of hash blocks produced.A longer piece of text to demonstrate how block size affects the number of hash blocks produced.A longer piece of text to demonstrate how block size affects the number of hash blocks produced.A longer piece of text to demonstrate how block size affects the number of hash blocks produced.A longer piece of text to demonstrate how block size affects the number of hash blocks produced.A longer piece of text to demonstrate how block size affects the number of hash blocks produced.A longer piece of text to demonstrate how block size affects the number of hash blocks produced.A longer piece of text to demonstrate how block size affects the number of hash blocks produced.A longer piece of text to demonstrate how block size affects the number of hash blocks produced.A longer piece of text to demonstrate how block size affects the number of hash blocks produced.A longer piece of text to demonstrate how block size affects the number of hash blocks produced.A longer piece of text to demonstrate how block size affects the number of hash blocks produced.A longer piece of text to demonstrate how block size affects the number of hash blocks produced.A longer piece of text to demonstrate how block size affects the number of hash blocks produced.A longer piece of text to demonstrate how block size affects the number of hash blocks produced."

for bs in [32, 64, 128]:
    writer.reset_hashes()
    hashes = writer.text_to_hashes(text, block_size=bs)
    print(f"block_size={bs:>4}  ->  {len(hashes)} hash block(s): {hashes}")

block_size=  32  ->  12 hash block(s): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
block_size=  64  ->  6 hash block(s): [0, 1, 2, 3, 4, 5]
block_size= 128  ->  3 hash block(s): [0, 1, 2]


## 7. Cross-Request Prefix Sharing

Because `MooncakeWriter` maintains a persistent hasher, hash IDs are consistent
across **separate** calls to `text_to_hashes`.  Two texts that share a token
prefix will receive the same leading hash IDs even when hashed at different times.

In [19]:
writer2 = MooncakeWriter("gpt2")

ids_a = writer2.text_to_hashes("The quick brown fox jumps over the lazy dog.")
ids_b = writer2.text_to_hashes("The quick brown fox jumps over the lazy dog.")
ids_c = writer2.text_to_hashes("A completely different sentence.")

print("Request A hashes:", ids_a)
print("Request B hashes:", ids_b)
print("Request C hashes:", ids_c)
print()
print(f"A and B share prefix: {ids_a[0] == ids_b[0] if ids_a and ids_b else 'N/A'}")
print(f"A and C share prefix: {ids_a[0] == ids_c[0] if ids_a and ids_c else 'N/A'}")
print(f"Unique hashes seen:   {writer2.hasher.get_stats()['total_hashes']}")

Request A hashes: [0]
Request B hashes: [0]
Request C hashes: [1]

A and B share prefix: True
A and C share prefix: False
Unique hashes seen:   2


## 8. Trace Capture and JSONL Output

In a proxy or live-capture scenario, use `capture()` to record each request as a
timestamped trace record.  You provide the timestamp yourself (in milliseconds) so
it reflects exactly when the request was received.

When you are done, call `write_trace()` to flush the buffer to a JSONL file that
aiperf can load directly as a `MooncakeTrace` dataset.

In [20]:
import time

trace_writer = MooncakeWriter("gpt2")

ts = int(time.time() * 1000)
trace_writer.capture("The quick brown fox jumps over the lazy dog.", timestamp_ms=ts, output_length=50)

ts = int(time.time() * 1000)
trace_writer.capture("The quick brown fox jumps over the lazy dog.", timestamp_ms=ts, output_length=30)

ts = int(time.time() * 1000)
trace_writer.capture("A completely different sentence.", timestamp_ms=ts, output_length=20)

print(f"Captured {len(trace_writer.traces)} trace records:\n")
for i, rec in enumerate(trace_writer.traces):
    print(f"  Record {i+1}: {rec}")

Captured 3 trace records:

  Record 1: {'timestamp': 1775181508803, 'input_length': 10, 'hash_ids': [0], 'output_length': 50}
  Record 2: {'timestamp': 1775181508803, 'input_length': 10, 'hash_ids': [0], 'output_length': 30}
  Record 3: {'timestamp': 1775181508803, 'input_length': 5, 'hash_ids': [1], 'output_length': 20}


In [21]:
trace_path = "my_trace.jsonl"
n = trace_writer.write_trace(trace_path)
print(f"Wrote {n} records to {trace_path}\n")

with open(trace_path) as f:
    print(f.read())

Wrote 3 records to my_trace.jsonl

{"timestamp": 1775181508803, "input_length": 10, "hash_ids": [0], "output_length": 50}
{"timestamp": 1775181508803, "input_length": 10, "hash_ids": [0], "output_length": 30}
{"timestamp": 1775181508803, "input_length": 5, "hash_ids": [1], "output_length": 20}

